# Chapitre 5 - Lab : réplication en streaming avec PostgreSQL

Les cours 1 et 2 ont simulé un leader et ses suiveurs. Ce lab observe la même chose sur
un vrai système : un primaire PostgreSQL 16 et deux répliques alimentées par l'expédition
du WAL (*streaming replication*). On y mesure le retard de réplication, on provoque une
lecture périmée, on implémente « lire ses écritures » avec un jeton de position, on
compare les niveaux de `synchronous_commit` et on voit un suiveur synchrone bloquer les
écritures, on lit le journal logique produit par `test_decoding`, et l'on termine par un
basculement qui perd des écritures confirmées puis par un split brain.

**Démarrer et arrêter les services** (depuis la racine du dépôt) :

```bash
docker compose -f docker/replication/compose.yml -p ddia_replication up -d
# ... exécuter le notebook (une seule fois : la fin modifie la topologie) ...
docker compose -f docker/replication/compose.yml -p ddia_replication down -v
```

Chaque réplique démarre par une copie de base du primaire (`pg_basebackup -R` :
instantané + position du journal, puis mode standby), exactement la recette de la
section *Setting Up New Followers*. Ports de l'hôte : primaire **10632**, `replica1`
**10633**, `replica2` **10634** (utilisateur, mot de passe et base : `ddia`).

**Plan.**

1. La topologie : un primaire, deux répliques en lecture seule.
2. Mesurer le retard de réplication.
3. Lecture périmée et « lire ses écritures » avec un jeton de LSN.
4. Réplication synchrone : coût et blocage.
5. Le journal logique.
6. Basculement : écritures perdues et split brain.

In [1]:
import subprocess
import threading
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg
from psycopg import errors

from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, finish, set_style, subtitle)
from ddia.timelines import SpaceTime

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.max_colwidth", 90)
ROOT = next(d for d in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (d / "docker/replication/compose.yml").exists())
COMPOSE = ["docker", "compose", "-f", str(ROOT / "docker/replication/compose.yml"), "-p", "ddia_replication"]
PORTS = {"primary": 10632, "replica1": 10633, "replica2": 10634}
DSN = {n: f"host=localhost port={p} user=ddia password=ddia dbname=ddia connect_timeout=5" for n, p in PORTS.items()}

def compose(*args):
    return subprocess.run(COMPOSE + list(args), check=True, capture_output=True, text=True)

def q(node, sql, params=None):
    with psycopg.connect(DSN[node], autocommit=True) as c:
        cur = c.execute(sql, params)
        return cur.fetchall() if cur.description else None

def attendre(node, timeout=60):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            return q(node, "SELECT 1")
        except psycopg.OperationalError:
            time.sleep(0.5)
    raise RuntimeError(f"{node} injoignable")

for n in PORTS:
    attendre(n)
print(q("primary", "SELECT version()")[0][0])

PostgreSQL 16.15 on x86_64-pc-linux-musl, compiled by gcc (Alpine 15.2.0) 15.2.0, 64-bit


## 1. La topologie (section *Leaders and Followers*)

Côté primaire, la vue `pg_stat_replication` liste les suiveurs connectés, leur état et
les positions (LSN) qu'ils ont reçues, écrites sur disque et rejouées. Côté répliques,
`pg_is_in_recovery()` vaut vrai : elles rejouent le journal et refusent les écritures.

In [2]:
for _ in range(60):
    if len(q("primary", "SELECT 1 FROM pg_stat_replication")) == 2:
        break
    time.sleep(1)
display(pd.DataFrame(q("primary", """SELECT application_name, state, sync_state, sent_lsn::text, replay_lsn::text
                                      FROM pg_stat_replication ORDER BY application_name"""),
                     columns=["suiveur", "état", "mode", "LSN envoyé", "LSN rejoué"]))
display(pd.DataFrame([(n, q(n, "SELECT pg_is_in_recovery()")[0][0]) for n in PORTS],
                     columns=["nœud", "en récupération (lecture seule)"]).set_index("nœud"))
q("primary", "DROP TABLE IF EXISTS mesures; CREATE TABLE mesures (id bigserial PRIMARY KEY, v int, t timestamptz DEFAULT now())")
time.sleep(1)
try:
    q("replica1", "INSERT INTO mesures (v) VALUES (1)")
except errors.ReadOnlySqlTransaction as e:
    print("écriture sur replica1 refusée :", e.diag.message_primary)

,suiveur,état,mode,LSN envoyé,LSN rejoué
0,replica1,streaming,async,0/4000000,0/4000000
1,replica2,streaming,async,0/4000000,0/4000000


,en récupération (lecture seule)
nœud,
primary,False
replica1,True
replica2,True


écriture sur replica1 refusée : cannot execute INSERT in a read-only transaction


Les deux suiveurs sont en `streaming`, en mode `async` (réglage par défaut) ; la table
créée sur le primaire est apparue sur les répliques par le journal, et une écriture
directe sur une réplique est refusée : les suiveurs sont en lecture seule pour les
clients.

> **À retenir.** PostgreSQL réalise la réplication à leader unique par expédition du WAL ;
> `pg_stat_replication` expose, pour chaque suiveur, les positions du journal qui
> permettent de mesurer son retard.